This file contains the code for homework 3 on classification.

In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [25]:
df = pd.read_csv("course_lead_scoring_2026.csv")

In [26]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [27]:
df.shape

(5000, 9)

In [28]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [29]:
df.info

<bound method DataFrame.info of          lead_source       industry employment_status       location  \
0     organic_search     technology          employed         europe   
1       social_media     technology          employed  south_america   
2           referral         retail          employed         europe   
3           paid_ads  manufacturing           student            NaN   
4           referral  manufacturing           student  north_america   
...              ...            ...               ...            ...   
4995  organic_search      education          employed           asia   
4996             NaN     technology          employed  north_america   
4997  organic_search        finance          employed           asia   
4998        referral     technology          employed         africa   
4999        referral     technology          employed         europe   

      annual_income  number_of_courses_viewed  interaction_count  lead_score  \
0           88160.0    

In [30]:
df.index

RangeIndex(start=0, stop=5000, step=1)

In [31]:
df.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='object')

In [32]:
df.describe()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
count,4631.000000,5000.000000,5000.000000,4965.000000,5000.000000
mean,54533.212049,2.089400,4.472600,0.494767,0.576600
std,21245.661710,1.188987,2.435906,0.199431,0.494147
min,12000.000000,0.000000,0.000000,0.010000,0.000000
25%,40458.500000,1.000000,3.000000,0.360000,0.000000
50%,55051.000000,2.000000,4.000000,0.490000,1.000000
75%,71252.500000,3.000000,6.000000,0.630000,1.000000
max,109886.000000,6.000000,13.000000,0.990000,1.000000


QUESTION 1

In [34]:
# Question 1: Find the mode of the industry column
print(df["industry"].mode())

0    technology
Name: industry, dtype: object


QUESTION 2

In [35]:

# Question 2: Correlation matrix for numerical features

numerical_features = df.select_dtypes(include="number")

correlation_matrix = numerical_features.corr()

print(correlation_matrix)


                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.137542   
number_of_courses_viewed       0.137542                  1.000000   
interaction_count              0.106375                  0.721609   
lead_score                     0.231396                  0.769352   
converted                      0.181752                  0.395038   

                          interaction_count  lead_score  converted  
annual_income                      0.106375    0.231396   0.181752  
number_of_courses_viewed           0.721609    0.769352   0.395038  
interaction_count                  1.000000    0.928712   0.448624  
lead_score                         0.928712    1.000000   0.490206  
converted                          0.448624    0.490206   1.000000  


SPLIT DATASET

In [36]:
%pip install scikit-learn
from sklearn.model_selection import train_test_split

# Split into full training data and test data
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
)


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
# Split full training data into training and validation data
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [38]:
print("Full training set:", df_full_train.shape)
print("Training set:", df_train.shape)
print("Validation set:", df_val.shape)
print("Test set:", df_test.shape)

Full training set: (4000, 9)
Training set: (3000, 9)
Validation set: (1000, 9)
Test set: (1000, 9)


In [39]:

# Extract the target labels
y_train = df_train["converted"].values
y_val = df_val["converted"].values
y_test = df_test["converted"].values

# Remove the target column from the feature DataFrames
df_train = df_train.drop(columns=["converted"])
df_val = df_val.drop(columns=["converted"])
df_test = df_test.drop(columns=["converted"])

# Verify the target is no longer in the feature DataFrames
print("converted in df_train:", "converted" in df_train.columns)
print("converted in df_val:", "converted" in df_val.columns)
print("converted in df_test:", "converted" in df_test.columns)


converted in df_train: False
converted in df_val: False
converted in df_test: False


QUESTION 3

In [40]:

# Check missing values in every column
print(df.isnull().sum())


lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


In [41]:

# Identify categorical and numerical columns
categorical_cols = df.select_dtypes(include=["object", "category"]).columns
numerical_cols = df.select_dtypes(include="number").columns.drop("converted")

# Fill missing values according to the assignment instructions
df[categorical_cols] = df[categorical_cols].fillna("NA")
df[numerical_cols] = df[numerical_cols].fillna(0.0)

# Verify that no missing values remain
print(df.isnull().sum())


lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


In [42]:

from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

# Save the target values
y_train = df_train["converted"].values
y_val = df_val["converted"].values
y_test = df_test["converted"].values

# Remove the target from the feature DataFrames
df_train = df_train.drop(columns=["converted"])
df_val = df_val.drop(columns=["converted"])
df_test = df_test.drop(columns=["converted"])


In [43]:

from sklearn.metrics import mutual_info_score

# Calculate mutual information for each categorical feature
categorical_features = [
    "industry",
    "location",
    "lead_source",
    "employment_status"
]

def mutual_info_converted(series):
    return round(mutual_info_score(y_train, series), 2)

mi_scores = df_train[categorical_features].apply(mutual_info_converted)

print(mi_scores.sort_values(ascending=False))


lead_source          0.03
employment_status    0.02
location             0.00
industry             0.00
dtype: float64


Question 4

In [21]:

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Convert DataFrames into lists of dictionaries
train_dicts = df_train.to_dict(orient="records")
val_dicts = df_val.to_dict(orient="records")

# One-hot encode categorical features
dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

# Train the logistic regression model
model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

# Predict conversion on the validation set
y_pred = model.predict(X_val)

# Calculate accuracy
accuracy = accuracy_score(y_val, y_pred)

print("Accuracy:", round(accuracy, 2))


Accuracy: 0.65


QUESTION 5

In [ ]:

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Baseline: train using all features
def train_and_evaluate(train_df, val_df):
    dv = DictVectorizer(sparse=False)

    X_train = dv.fit_transform(
        train_df.to_dict(orient="records")
    )
    X_val = dv.transform(
        val_df.to_dict(orient="records")
    )

    model = LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    return accuracy_score(y_val, y_pred)


baseline_accuracy = train_and_evaluate(df_train, df_val)

features_to_remove = [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count"
]

for feature in features_to_remove:
    train_without = df_train.drop(columns=[feature])
    val_without = df_val.drop(columns=[feature])

    accuracy_without = train_and_evaluate(
        train_without, val_without
    )

    difference = baseline_accuracy - accuracy_without

    print(
        f"{feature}: "
        f"accuracy_without={accuracy_without:.3f}, "
        f"difference={difference:.3f}"
    )


QUESTION 6

In [ ]:

# Question 6: Compare different values of C

C_values = [0.000001, 0.00001, 0.0001, 0.001]

for C in C_values:
    model = LogisticRegression(
        solver="liblinear",
        C=C,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    accuracy = accuracy_score(y_val, y_pred)

    print(f"C = {C}, Accuracy = {accuracy:.3f}")
